In [32]:
import pandas as pd

In [33]:
df = pd.read_csv("dataset\jarvis_intents_2000.csv");
print( df.sample(10))

                      command           intent
1491             start github      open_github
982   can you math calculator  open_calculator
167              current time         get_time
965        jarvis calc for me  open_calculator
819            start calc app  open_calculator
54            what's the time         get_time
17        current time for me         get_time
752        can you open notes     open_notepad
1307   linkedin please for me    open_linkedin
190       current time for me         get_time


In [34]:
df['intent'].value_counts()

intent
get_time           200
search_google      200
search_youtube     200
open_notepad       200
open_calculator    200
open_whatsapp      200
open_linkedin      200
open_github        200
open_spotify       200
exit               200
Name: count, dtype: int64

In [35]:
df.describe()


,command,intent
count,2000,2000
unique,650,10
top,search on google,get_time
freq,14,200


In [36]:
df['intent_number'] = df['intent'].map({
    'get_time': 0,
    'search_google': 1,
    'search_youtube': 2,
    'open_notepad': 3,
    'open_calculator': 4,
    'open_whatsapp': 5,
    'open_linkedin': 6,
    'open_github': 7,
    'open_spotify': 8,
    'exit': 9
})
print( df.sample(10))

                        command          intent  intent_number
1884     please close assistant            exit              9
229               search online   search_google              1
478       youtube lookup for me  search_youtube              2
1457         open github please     open_github              7
225           look up on google   search_google              1
1100     please whatsapp please   open_whatsapp              5
1523               start github     open_github              7
757   start writing tool please    open_notepad              3
446   hey jarvis youtube search  search_youtube              2
1052       open whatsapp for me   open_whatsapp              5


In [37]:
import spacy
from spacy.lang.en.stop_words import STOP_WORDS

stopwords = list(STOP_WORDS)
print(stopwords)


['otherwise', 'yourself', 'still', 'this', 'give', "'s", 'full', 'rather', 'of', 'various', 'down', 'from', 'latter', 'some', '’m', 'mostly', 'quite', 'on', 'hereupon', 'empty', 'beyond', 'because', 'it', 'off', 'how', 'am', 'if', 're', 'first', 'keep', "'m", 'anyone', 'part', '’s', 'often', 'whereafter', 'done', 'one', 'most', 'whereby', 'show', 'where', 'anything', 'whereas', 'used', 'except', 'thence', 'enough', 'n’t', 'not', 'another', 'cannot', 'three', 'only', 'alone', 'seems', 'could', 'above', 'same', 'been', 'along', 'thereafter', '’ll', 'serious', 'your', 'hence', 'forty', 'fifty', 'seem', 'at', '‘m', 'ours', 'nobody', 'say', 'put', 'but', 'mine', 'thereupon', 'twenty', 'thus', 'meanwhile', 'again', 'always', 'due', 'move', 'her', 'did', 'several', 'sixty', 'wherever', 'hereafter', 'with', 'four', 'n‘t', 'can', 'now', 'back', 'whom', 'made', 'seemed', 'nor', 'few', 'were', 'had', 'have', 'she', '’re', 'once', 'everything', 'here', 'besides', 'five', 'please', 'everyone', 'amo

In [38]:
df['command_without_stopwords'] = df['command'].apply(lambda x: ' '.join([word for word in x.split() if word not in stopwords]))

In [39]:
df.head(10)

,command,intent,intent_number,command_without_stopwords
0,jarvis time,get_time,0,jarvis time
1,time now please,get_time,0,time
2,what's the time,get_time,0,what's time
3,hey jarvis current time,get_time,0,hey jarvis current time
4,hey jarvis what's the time,get_time,0,hey jarvis what's time
5,what's the time,get_time,0,what's time
6,what time is it now,get_time,0,time
7,time please please,get_time,0,time
8,what's the time for me,get_time,0,what's time
9,hey jarvis what's the time,get_time,0,hey jarvis what's time


In [40]:
from spacy.lang.en.stop_words import STOP_WORDS
stopwords = list(STOP_WORDS)
df['command_without_stopwords'] = df['command'].apply(
    lambda x: ' '.join([word for word in x.split() if word.lower() not in stopwords])
)

# 4️⃣ Convert to embeddings
nlp = spacy.load('en_core_web_md')
df['command_vector'] = df['command_without_stopwords'].apply(lambda x: nlp(x).vector)

In [41]:
df.sample(10)


,command,intent,intent_number,command_without_stopwords,command_vector
9,hey jarvis what's the time,get_time,0,hey jarvis what's time,"[-0.68844, 0.214672, -0.1294178, -0.238977, 0...."
851,run calculator,open_calculator,4,run calculator,"[-0.69964504, 0.40008, -0.298525, 0.005820006,..."
1476,check github please,open_github,7,check github,"[-0.829885, 0.0026325006, -0.24982001, 0.22575..."
905,open calc for me,open_calculator,4,open calc,"[-0.66603994, 0.19411999, -0.2113365, 0.316775..."
1962,can you close jarvis,exit,9,close jarvis,"[-0.65575504, 0.255045, 0.080754995, 0.072845,..."
1642,start spotify now,open_spotify,8,start spotify,"[-0.73223, 0.43322, -0.1534, -0.016059995, -0...."
385,google query for me,search_google,1,google query,"[-0.679885, 0.57078004, -0.32488, 0.20802, -0...."
1607,start spotify,open_spotify,8,start spotify,"[-0.73223, 0.43322, -0.1534, -0.016059995, -0...."
1296,linkedin profile,open_linkedin,6,linkedin profile,"[-0.856025, 0.79717, -0.27325, -0.0053199977, ..."
892,math calculator,open_calculator,4,math calculator,"[-0.660915, 0.43484002, -0.119040504, 0.320460..."


In [42]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df.command_vector.values,
    df.intent_number,
    test_size=0.2,
    random_state=42
)

In [43]:
X_test.shape

(400,)

In [44]:
import numpy as np
X_train_2d = np.stack(X_train)
X_test_2d = np.stack(X_test)

In [45]:
X_train_2d

array([[-0.73016   ,  0.31736   , -0.283985  , ..., -0.03101501,
         0.2741315 , -0.145015  ],
       [-0.709615  ,  0.50777   , -0.354875  , ..., -0.23804998,
        -0.0193395 ,  0.18947649],
       [-0.76467997,  0.38349664, -0.22445333, ..., -0.20174332,
         0.22660601, -0.17488568],
       ...,
       [-0.66578   ,  0.38624   , -0.15455   , ...,  0.50593   ,
         0.59029   , -0.43495   ],
       [-0.701565  ,  0.08198425, -0.1117125 , ..., -0.15189001,
        -0.31121427,  0.40516573],
       [-0.74075496,  0.01431   ,  0.01219501, ..., -0.07066001,
        -0.2757685 ,  0.63971   ]], shape=(1600, 300), dtype=float32)

In [ ]:
import os
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report


MODEL_DIR = os.path.join(os.getcwd(), "models")

os.makedirs(MODEL_DIR, exist_ok=True)


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_2d)
X_test_scaled = scaler.transform(X_test_2d)


base_models = [
    ('lr', LogisticRegression(max_iter=1000)),
    ('rf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
    ('svm', SVC(kernel='linear', probability=True, random_state=42))
]


stack_clf = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
    n_jobs=-1
)


stack_clf.fit(X_train_scaled, y_train)

y_pred = stack_clf.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred) * 100)
print(classification_report(y_test, y_pred))


model_path = os.path.join(MODEL_DIR, "jarvis_intent_model_stack.pkl")
scaler_path = os.path.join(MODEL_DIR, "jarvis_scaler_stack.pkl")

joblib.dump(stack_clf, model_path)
joblib.dump(scaler, scaler_path)

print("Model saved at:", model_path)
print("Scaler saved at:", scaler_path)

NameError: name '__file__' is not defined

In [ ]:
import joblib
import spacy
from spacy.lang.en.stop_words import STOP_WORDS
import numpy as np

# 1️⃣ Load saved model and scaler
stack_clf = joblib.load("jarvis_intent_model_stack.pkl")
scaler = joblib.load("jarvis_scaler_stack.pkl")

# 2️⃣ Load spaCy model
nlp = spacy.load('en_core_web_md')

# 3️⃣ Define stopwords
stopwords = list(STOP_WORDS)

# 4️⃣ Map intent numbers back to intent names
intent_map = {
    0: 'get_time',
    1: 'search_google',
    2: 'search_youtube',
    3: 'open_notepad',
    4: 'open_calculator',
    5: 'open_whatsapp',
    6: 'open_linkedin',
    7: 'open_github',
    8: 'open_spotify',
    9: 'exit'
}

# 5️⃣ Function to preprocess and predict intent
def predict_intent(command):
    # Remove stopwords
    command_clean = ' '.join([word for word in command.split() if word.lower() not in stopwords])
    
    # Convert to embedding
    command_vector = nlp(command_clean).vector.reshape(1, -1)
    
    # Scale vector
    command_scaled = scaler.transform(command_vector)
    
    # Predict intent number
    intent_number = stack_clf.predict(command_scaled)[0]
    
    # Map number to intent name
    return intent_map[intent_number]

# 6️⃣ Test commands
test_commands = [
    "Open Spotify",
    "What time is it?",
    "Launch Notepad",
    "Search Python tutorials on Google",
    "Open my GitHub profile",
    "Exit Jarvis",
    "Tell me the current time",
    "Close the assistant"
]

# 7️⃣ Run predictions
for cmd in test_commands:
    intent_name = predict_intent(cmd)
    print(f"Command: '{cmd}' -> Predicted Intent: {intent_name}")
    print (accuracy_score)


Command: 'Open Spotify' -> Predicted Intent: open_spotify
<function accuracy_score at 0x000001EA70920A60>
Command: 'What time is it?' -> Predicted Intent: get_time
<function accuracy_score at 0x000001EA70920A60>
Command: 'Launch Notepad' -> Predicted Intent: open_notepad
<function accuracy_score at 0x000001EA70920A60>
Command: 'Search Python tutorials on Google' -> Predicted Intent: search_google
<function accuracy_score at 0x000001EA70920A60>
Command: 'Open my GitHub profile' -> Predicted Intent: open_github
<function accuracy_score at 0x000001EA70920A60>
Command: 'Exit Jarvis' -> Predicted Intent: exit
<function accuracy_score at 0x000001EA70920A60>
Command: 'Tell me the current time' -> Predicted Intent: get_time
<function accuracy_score at 0x000001EA70920A60>
Command: 'Close the assistant' -> Predicted Intent: exit
<function accuracy_score at 0x000001EA70920A60>
